# Machine Vision & Deep Learning - Labsheet v1.0.2
**Asst. Prof. Dr. Sarucha Yanyong | KMITL**

เอกสาร Labsheet นี้ออกแบบมาเพื่อให้ผู้เรียนสามารถทดลองเขียนโค้ดตามสไลด์บรรยาย (`TripleHelixMV_v1.0.pptx`) ได้จริง โดยโค้ดถูกแบ่งเป็น 1 โค้ดบล็อกต่อ 1 สไลด์ เพื่อให้สามารถนำไปรันบน **Google Colab** ได้อย่างสะดวก

---

## การเตรียมตัวบน Google Colab

**ขั้นที่ 1 — อัปโหลดไฟล์ภาพ `part.jpg`**

ทุกโค้ดบล็อกในเอกสารนี้อ่านภาพจากไฟล์ชื่อ `part.jpg` ดังนั้นต้องอัปโหลดไฟล์นี้เข้า Colab ก่อน มิฉะนั้นจะขึ้นข้อความ `Error: ไม่พบไฟล์ภาพ`

วิธีอัปโหลด: คลิกไอคอน **โฟลเดอร์ (Files)** ที่แถบด้านซ้ายของ Colab → กดปุ่ม **Upload to session storage** → เลือกไฟล์ `part.jpg` จากเครื่องของตนเอง

ไฟล์เดียวขนาดเล็ก ไม่จำเป็นต้อง mount Google Drive แต่อย่างใด (ข้อควรทราบ: ไฟล์ที่อัปโหลดจะหายเมื่อ session ของ Colab ถูกปิด ถ้า runtime หลุดให้อัปโหลดใหม่)

**ขั้นที่ 2 — ติดตั้งไลบรารี** (รันโค้ดบล็อกถัดไป)


In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Colab
!pip install opencv-python numpy matplotlib



---

## 1. Digital Image Fundamentals (พื้นฐานภาพดิจิทัล)

### Slide 11: การอ่านภาพและเข้าถึงพิกเซล
*คำอธิบาย:* ใน OpenCV ภาพดิจิทัลจะถูกอ่านเข้ามาเป็นเมทริกซ์ 3 มิติ (ความสูง x ความกว้าง x จำนวนช่องสี) โดยค่าเริ่มต้นจะเรียงช่องสีแบบ BGR (Blue, Green, Red) การเข้าใจโครงสร้างพิกเซลเป็นก้าวแรกในการประมวลผลภาพ

**ฟังก์ชันที่สำคัญ:**
- `cv2.imread()`: ใช้สำหรับโหลดไฟล์ภาพจากดิสก์เข้ามาเก็บเป็นตัวแปรโครงสร้างเมทริกซ์ (NumPy Array) ในหน่วยความจำโปรแกรม

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| flag ของ `cv2.imread(path, flag)` | `cv2.IMREAD_GRAYSCALE` | ได้ภาพเทาทันที `shape` เหลือ 2 มิติ (H, W) ประหยัดหน่วยความจำ 3 เท่า |
| | `cv2.IMREAD_UNCHANGED` | เก็บช่อง alpha (ความโปร่งใส) ไว้ด้วย ได้ 4 ช่องสำหรับ PNG |
| พิกัดใน `img[y, x]` | `img[h//2, w//2]` | อ่านค่าพิกเซลกลางภาพ (บนชิ้นงาน) แทนมุมบนซ้ายที่เป็นพื้นหลัง |

> **ข้อควรระวัง:** ถ้าพิมพ์ชื่อไฟล์ผิด `cv2.imread()` จะคืนค่า `None` เงียบๆ ไม่ฟ้อง error — โค้ดจึงต้องเช็ค `if img is None` ทุกครั้ง


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is None:
    print("Error: ไม่พบไฟล์ภาพ")
else:
    print("Image Shape:", img.shape)
    print("Data Type:", img.dtype)
    print("Pixel (0,0):", img[0, 0])



### Slide 12: การย่อขนาดภาพ
*คำอธิบาย:* การย่อภาพ (Resizing) ช่วยลดจำนวนพิกเซลที่ต้องประมวลผล ทำให้โปรแกรมทำงานได้เร็วขึ้นและประหยัดหน่วยความจำ แลกกับรายละเอียดของภาพ (Fine details) ที่อาจสูญเสียไป

**ฟังก์ชันและคำสั่งที่สำคัญ:**
- `h, w = img.shape[:2]`: คำสั่งดึงค่า ความสูง (h) และความกว้าง (w) ของภาพจากแอตทริบิวต์ `.shape`
- `cv2.resize()`: ใช้ปรับเปลี่ยนขนาดของภาพ โดยสามารถระบุขนาดใหม่ (Width, Height) และเลือกเทคนิคการประมาณค่าพิกเซลใหม่ (Interpolation) ได้ (เช่น `cv2.INTER_AREA` เหมาะสำหรับการย่อภาพโดยไม่เกิดรอยหยัก)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| ตัวหารขนาด `(w//4, h//4)` | `//2` | ภาพเหลือ 1/4 ของจำนวนพิกเซล ประมวลผลเร็วขึ้นราว 4 เท่า รายละเอียดยังดี |
| | `//8` | เร็วขึ้นมาก แต่ฟีเจอร์เล็ก เช่น ตัวอักษร `LOT A17` จะอ่านไม่ออก |
| `interpolation` | `cv2.INTER_AREA` | **มาตรฐานสำหรับการย่อ** — เฉลี่ยพิกเซล ไม่เกิดรอยหยัก |
| | `cv2.INTER_NEAREST` | เร็วที่สุดแต่ขอบเป็นขั้นบันได เหมาะกับภาพ mask/binary ที่ห้ามให้ค่าถูกเฉลี่ย |
| | `cv2.INTER_CUBIC` | เหมาะกับการ **ขยาย** ภาพ ให้ขอบเนียนกว่า linear แต่ช้ากว่า |

> **หลักในงานจริง:** ย่อภาพช่วยให้เร็วขึ้น แต่ต้องเหลือพิกเซลคลุมฟีเจอร์ที่เล็กที่สุดอย่างน้อย 3–5 พิกเซล ไม่เช่นนั้นจะตรวจไม่เจอ


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    h, w = img.shape[:2]
    print(f"Resolution: {w} x {h} = {w*h} px")

    small = cv2.resize(img, (w//4, h//4), interpolation=cv2.INTER_AREA)
    cv2_imshow(img)
    print("Small:")
    cv2_imshow(small)



### Slide 13: การลดระดับ Bit Depth
*คำอธิบาย:* ภาพทั่วไปใช้ 8-bit ต่อช่องสี (256 ระดับ) การลองลด Bit depth ให้เหลือ 4-bit (16 ระดับ) ช่วยให้เห็นภาพรวมของการเกิด Color Banding และเข้าใจว่าทำไมระบบ Vision บางประเภทที่ต้องการความแม่นยำสูงจึงอาจต้องใช้กล้อง 10-bit หรือ 12-bit

**คำสั่งที่สำคัญ:**
- `(img // 16) * 16`: เป็นทริคทางคณิตศาสตร์เพื่อลดจำนวนระดับสี โดยการหารปัดเศษทิ้ง (`//`) ให้ค่าสีแบ่งเป็นกลุ่มกว้างๆ ก่อน แล้วค่อยคูณกลับเพื่อขยายกลับไปสเกลเดิม ทำให้ค่าพิกเซลถูกจัดกลุ่มหยาบขึ้นเกิดเป็น Effect ของภาพแบบ Bit depth ต่ำ

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| ตัวหารใน `(img // 16) * 16` | `// 4 * 4` | เหลือ 64 ระดับ (6-bit) — แทบแยกจากภาพต้นฉบับไม่ออก |
| | `// 32 * 32` | เหลือ 8 ระดับ (3-bit) — เห็นแถบสี (Color Banding) ชัดเจน |
| | `// 64 * 64` | เหลือ 4 ระดับ (2-bit) — ภาพแตกเป็นหย่อมสีใหญ่ๆ |

**สูตร:** จำนวนระดับสีที่เหลือ = 256 ÷ ตัวหาร → ตัวหารยิ่งมาก ระดับสียิ่งน้อย ภาพยิ่งหยาบ

> **โยงกับงานจริง:** นี่คือเหตุผลที่งานวัดความเข้มละเอียด (เช่น ตรวจรอยจางบนผิว) ต้องใช้กล้อง 10-bit หรือ 12-bit แทน 8-bit


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    print("Original Dtype:", img.dtype)
    q = (img // 16) * 16
    print("4-bit depth simulation:")
    cv2_imshow(q)



### Slide 14: การแยกแชนเนล RGB
*คำอธิบาย:* บางครั้งวัตถุที่เราต้องการตรวจสอบอาจมีความแตกต่างชัดเจนในช่องสีใดสีหนึ่ง (เช่น รอยตำหนิสีแดงจะเห็นชัดในช่อง R) การแยกช่องสีจึงเป็นเทคนิคพื้นฐานที่ช่วยทำให้ฟีเจอร์เด่นขึ้นโดยไม่ต้องพึ่งอัลกอริทึมซับซ้อน

**ฟังก์ชันที่สำคัญ:**
- `cv2.cvtColor()`: ใช้แปลงระบบสีของภาพ เช่น ในโค้ดคือเปลี่ยนจาก BGR (ค่าเริ่มต้นของ OpenCV) เป็น RGB เพื่อให้สีเรียงลำดับตรงตามมาตรฐานทั่วไป
- `cv2.split()`: ใช้แยกภาพสี 1 ภาพ (3 แชนเนล) ออกเป็นเมทริกซ์เดี่ยวภาพขาวดำ (Grayscale) 3 เมทริกซ์ตามข้อมูลแต่ละช่องสี (R, G, B)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| ช่องสีที่นำไปแสดง | `cv2_imshow(r)` | เห็นเฉพาะ **ช่อง R** — ปุ่มแดงและปุ่มเหลืองสว่าง ปุ่มเขียว/น้ำเงินมืด |
| | `cv2_imshow(g)` | เห็นเฉพาะ **ช่อง G** — ปุ่มเขียวและปุ่มเหลืองสว่าง ปุ่มแดง/น้ำเงินมืด |
| | `cv2_imshow(b)` | เห็นเฉพาะ **ช่อง B** — ปุ่มน้ำเงินสว่าง ที่เหลือมืด |
| ลำดับช่องสี | `b, g, r = cv2.split(img)` | ถ้าแยกจากภาพ BGR ต้นฉบับ (ไม่แปลงเป็น RGB ก่อน) ลำดับตัวแปรต้องเป็น b, g, r มิฉะนั้นค่าจะสลับกัน |

> **สำคัญ:** `cv2_imshow(g)` แสดงผลออกมาเป็น **ภาพขาวดำ ไม่ใช่ภาพสีเขียว** เพราะ `cv2.split()` คืนค่าเป็นเมทริกซ์ 2 มิติ (H, W) ที่ไม่มีข้อมูลสีเหลืออยู่ — ความสว่างในภาพนั้นหมายถึง "ปริมาณของสีเขียวในพิกเซลนั้น" (ขาว = เขียวเข้มจัด, ดำ = ไม่มีเขียวเลย)

**ค่าจริงที่จุดกลางปุ่มแต่ละสีในภาพ `part.jpg`**

| จุด | R | G | B | สังเกต |
|---|---:|---:|---:|---|
| ปุ่มเขียว | 1 | 180 | 0 | สว่างเฉพาะช่อง G |
| ปุ่มแดง | 220 | 1 | 0 | สว่างเฉพาะช่อง R |
| ปุ่มน้ำเงิน | 19 | 72 | 214 | สว่างเฉพาะช่อง B |
| ปุ่มเหลือง | 223 | 209 | 0 | สว่างทั้งช่อง R และ G (เหลือง = แดง + เขียว) |

> **หลักในงานจริง:** ถ้าตำหนิมีสีเด่นชัด การเลือกดูแค่ช่องสีเดียวมักได้ contrast ดีกว่าการแปลงเป็น grayscale ทั้งภาพ (ซึ่งเฉลี่ยทุกช่องรวมกันจนสีจาง)


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is None:
    print("Error: ไม่พบไฟล์ภาพ (กรุณาอัปโหลด part.jpg เข้า Colab ก่อน)")
else:
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    r, g, b = cv2.split(rgb)

    print("Mean R:", r.mean())
    print("Mean G:", g.mean())
    print("Mean B:", b.mean())

    # แต่ละช่องเป็นเมทริกซ์ 2 มิติ (H, W) จึงแสดงผลออกมาเป็น "ภาพขาวดำ" ไม่ใช่ภาพสีเขียว/แดง
    # ความสว่าง = ปริมาณของสีนั้นในแต่ละพิกเซล (ขาว = มีมาก, ดำ = มีน้อย)
    print("Channel G shape:", g.shape, "(2 มิติ = ภาพขาวดำ)")

    print("ช่อง G : ปุ่มเขียวสว่าง ปุ่มแดง/น้ำเงินมืด")
    cv2_imshow(g)

    # ลองสลับดูช่องอื่น
    # cv2_imshow(r)   # ปุ่มแดงและปุ่มเหลืองสว่าง
    # cv2_imshow(b)   # ปุ่มน้ำเงินสว่าง




### Slide 15: การคัดแยกสีด้วย HSV (ส่วนที่ 1: การสร้าง Mask)
*คำอธิบาย:* ในส่วนที่ 1 นี้ เราแปลงภาพเป็นโมเดลสี HSV (Hue, Saturation, Value) เพราะมันทนทานต่อแสงเงาได้ดีกว่า RGB จากนั้นใช้คำสั่ง `cv2.inRange()` เพื่อกรองเอาเฉพาะช่วงสีที่เราสนใจ (เช่น สีเขียว) ผลลัพธ์ที่ได้ในขั้นตอนนี้คือ "Mask" ซึ่งเป็นภาพขาวดำ (ขาว = บริเวณที่สีตรงกับเงื่อนไข, ดำ = ไม่ตรง) คล้ายกับการทำสเตนซิล (Stencil) ก่อนจะนำไปตัดภาพจริงในสไลด์ถัดไป

**ฟังก์ชันที่สำคัญ:**
- `cv2.cvtColor(..., cv2.COLOR_BGR2HSV)`: ทำการแปลงภาพเป็น HSV เพื่อให้ง่ายต่อการระบุเฉดสี (อิงตามค่า H)
- `cv2.inRange()`: ใช้ตรวจสอบว่าพิกเซลแต่ละจุดมีค่าสีอยู่ในช่วง `lo` ถึง `hi` ที่กำหนดหรือไม่ แล้วสร้างออกมาเป็นภาพหน้ากาก (Mask) สีขาวดำตามเงื่อนไข

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

`lo, hi = (H, S, V)` — ใน OpenCV ค่า **H = 0–179**, **S และ V = 0–255**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| ช่วง H (เฉดสี) | `(35, 80, 80)` → `(85, 255, 255)` | จับ **สีเขียว** (ค่าที่ตั้งไว้ในโค้ด) |
| | `(0, 120, 70)` → `(10, 255, 255)` | จับ **สีแดง** (แดงคร่อมปลายสเกล อาจต้องทำ 2 ช่วงแล้ว `cv2.bitwise_or` รวมกัน: 0–10 และ 170–179) |
| | `(100, 120, 70)` → `(130, 255, 255)` | จับ **สีน้ำเงิน** |
| | `(20, 120, 120)` → `(35, 255, 255)` | จับ **สีเหลือง** |
| S ต่ำสุด (ตัวที่ 2 ของ `lo`) | ลดลงเช่น `30` | รับสีซีด/สีเทาเข้ามาด้วย mask รกขึ้น |
| | เพิ่มขึ้นเช่น `150` | เอาเฉพาะสีจัดจริงๆ mask สะอาดแต่อาจขาดบริเวณที่โดนแสงจ้า |
| V ต่ำสุด (ตัวที่ 3 ของ `lo`) | เพิ่มขึ้นเช่น `120` | ตัดบริเวณเงามืดออก ป้องกันเงาถูกนับเป็นวัตถุ |

> **วิธีหาค่าเร็วๆ:** พิมพ์ `print(hsv[y, x])` ที่พิกัดกลางวัตถุที่ต้องการ แล้วตั้งช่วง H บวก/ลบ ประมาณ 10 จากค่านั้น


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lo, hi = (35, 80, 80), (85, 255, 255)
    mask = cv2.inRange(hsv, lo, hi)

    cv2_imshow(mask)



### Slide 16: การคัดแยกสีด้วย HSV (ส่วนที่ 2 - ซ้อนทับ)
*คำอธิบาย:* เมื่อเราได้แผ่น Mask (สเตนซิลขาวดำ) จากส่วนที่ 1 แล้ว ในขั้นตอนนี้เราจะใช้คำสั่ง `cv2.bitwise_and()` เพื่อเอา Mask มาทาบทับลงบนภาพสีต้นฉบับ ผลลัพธ์คือภาพสีปกติ แต่จะถูก "ตัด" มาเฉพาะบริเวณที่ตรงกับสีขาวของ Mask เท่านั้น ส่วนบริเวณอื่นจะกลายเป็นสีดำทั้งหมด

**ฟังก์ชันที่สำคัญ:**
- `cv2.bitwise_and()`: ทำตรรกะทางคอมพิวเตอร์ (AND เชิงบิต) ระหว่างภาพสองภาพ โดยถ้าระบุพารามิเตอร์ `mask` เสริมเข้าไป ฟังก์ชันนี้จะทำการคัดลอกค่าพิกเซลของภาพต้นฉบับมาแสดงผล เฉพาะในพิกัดที่ Mask มีค่าเป็นสีขาวเท่านั้น

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `mask` ที่ส่งเข้า `bitwise_and` | `cv2.bitwise_not(mask)` | ได้ผลกลับด้าน — เห็นทุกอย่าง **ยกเว้น** สีที่เลือก ใช้ตรวจว่า mask กินพื้นที่เกินไปหรือไม่ |
| ภาพต้นทางที่นำมา and | `cv2.bitwise_and(img, img, mask=mask)` | ตัดเฉพาะบริเวณ mask จากภาพสีต้นฉบับ (ค่าที่ใช้ในโค้ด) |
| ขั้นตอนก่อนหน้า | เติม `cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)` ก่อน and | ลบจุดรบกวนเล็กๆ ใน mask ให้ผลลัพธ์สะอาดขึ้น (เทคนิคจากสไลด์ 58) |

> **หลักในงานจริง:** ลำดับที่ใช้เสมอคือ **แปลง HSV → inRange → ทำความสะอาด mask ด้วย Morphology → นำไปนับ/วัด** ข้ามขั้นทำความสะอาดมักทำให้นับชิ้นงานเกินจากจุด noise


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lo, hi = (35, 80, 80), (85, 255, 255)
    mask = cv2.inRange(hsv, lo, hi)
    out = cv2.bitwise_and(img, img, mask=mask)

    cv2_imshow(out)



### Slide 18: Image Transformation (Affine)
*คำอธิบาย:* ในงานอุตสาหกรรม ชิ้นงานอาจไม่ได้วางตรงตำแหน่งเดิมเสมอไป การทำ Affine Transform (หมุนและเลื่อนภาพ) จึงเป็นพื้นฐานในการทำ Image Alignment เพื่อจัดตำแหน่งภาพก่อนที่จะทำการวัดหรือตรวจสอบขั้นตอนถัดไป

**ฟังก์ชันที่สำคัญ:**
- `cv2.getRotationMatrix2D()`: ใช้สร้างเมทริกซ์คณิตศาสตร์สำหรับคำนวณการหมุนภาพ โดยเราต้องระบุ พิกัดจุดหมุน (ศูนย์กลาง), องศาการหมุน, และสัดส่วนการย่อขยาย
- `cv2.warpAffine()`: ใช้ประยุกต์เมทริกซ์การแปลง (เช่น หมุน, เลื่อน ที่สร้างไว้) เข้ากับรูปภาพจริงเพื่อคำนวณและสร้างภาพผลลัพธ์ที่เปลี่ยนพิกัดไปแล้ว

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| องศาใน `getRotationMatrix2D(center, 45, 1.0)` | `90` / `-30` | หมุนทวนเข็ม 90° / ตามเข็ม 30° (ค่าบวก = ทวนเข็มนาฬิกา) |
| scale (ตัวที่ 3) | `0.5` | หมุนพร้อมย่อครึ่ง ทำให้เห็นภาพครบไม่โดนตัดมุม |
| จุดหมุน (ตัวแรก) | `(0, 0)` | หมุนรอบมุมบนซ้ายแทนกลางภาพ ภาพจะหลุดเฟรมเกือบหมด |
| ค่าใน `T = [[1,0,50],[0,1,30]]` | เลข `50` | เลื่อนขวา 50 px (ติดลบ = เลื่อนซ้าย) |
| | เลข `30` | เลื่อนลง 30 px (ติดลบ = เลื่อนขึ้น) |
| ขนาดผลลัพธ์ `warpAffine(img, M, (w, h))` | `(int(w*1.5), int(h*1.5))` | ขยายผืนผ้าใบ ไม่ให้มุมภาพที่หมุนออกไปถูกตัดทิ้ง |

> **โยงกับงานจริง:** เทคนิคนี้คือหัวใจของ **Image Alignment** — จับมุมเอียงของชิ้นงานด้วย `minAreaRect` (สไลด์ 69) แล้วหมุนกลับให้ตรงก่อนวัด จะได้ค่าที่เทียบกันได้ทุกชิ้น


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    h, w = img.shape[:2]

    M = cv2.getRotationMatrix2D((w/2, h/2), 45, 1.0)
    rot = cv2.warpAffine(img, M, (w, h))

    T = np.float32([[1, 0, 50], [0, 1, 30]])
    sh = cv2.warpAffine(img, T, (w, h))

    print("Rotated 45:")
    cv2_imshow(rot)
    print("Translated:")
    cv2_imshow(sh)



---

## 2. Lens & Optics (เลนส์และทัศนศาสตร์)

### Slide 41: การคำนวณ Focal Length
*คำอธิบาย:* ในการออกแบบระบบ Machine Vision เราจำเป็นต้องเลือกเลนส์ที่มีระยะโฟกัส (Focal Length) สอดคล้องกับขนาดเซนเซอร์กล้อง (Sensor Size) และมุมมองภาพ (FOV) ที่ระยะการทำงาน (Working Distance) เพื่อให้เห็นชิ้นงานได้ครอบคลุมพอดี

**การคำนวณที่สำคัญ:**
- เป็นการจำลองการเขียนฟังก์ชันเพื่อคำนวณสูตรเรขาคณิตทั่วไปเพื่อหาระยะโฟกัสทางทฤษฎี `f = (Sensor Width / FOV) * Working Distance`

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

สูตร `f = (Sensor Width / FOV) × WD` — ลองแทนค่าแล้วสังเกตทิศทางการเปลี่ยนแปลง

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `WD` (ระยะทำงาน) 500 mm | `1000` | f = 112.6 mm — **ถอยกล้องไกลขึ้น 2 เท่า ต้องใช้เลนส์ยาวขึ้น 2 เท่า** (แปรผันตรง) |
| `FOV` (พื้นที่ที่ต้องเห็น) 100 mm | `200` | f = 28.2 mm — **อยากเห็นกว้างขึ้น ต้องใช้เลนส์สั้นลง** (แปรผกผัน) |
| | `50` | f = 112.6 mm — ซูมเข้าไปดูพื้นที่แคบลง ต้องใช้เลนส์ยาวขึ้น |
| `sensor_w` (ขนาดเซนเซอร์) 11.26 mm | `7.06` (เซนเซอร์ 1/1.8") | f = 35.3 mm — เซนเซอร์เล็กลงใช้เลนส์สั้นลงที่ FOV เท่ากัน |

**ตรวจงานต่อ:** ขนาดต่อพิกเซล = FOV ÷ จำนวนพิกเซลด้านนั้น เช่น FOV 100 mm บนเซนเซอร์ 2048 px → 0.049 mm/px ถ้าต้องตรวจตำหนิขนาด 0.5 mm จะได้ราว 10 พิกเซล ถือว่าเพียงพอ (ควรมีอย่างน้อย 3–5 พิกเซลต่อฟีเจอร์)


In [ ]:
def calculate_focal_length(sensor_width_mm, working_distance_mm, fov_mm):
    return (sensor_width_mm / fov_mm) * working_distance_mm

sensor_w = 11.26
WD = 500.0
FOV = 100.0

f_computed = calculate_focal_length(sensor_w, WD, FOV)
print(f"Computed Focal Length: {round(f_computed, 1)} mm")



### Slide 42: การเลือกเลนส์มาตรฐาน
*คำอธิบาย:* เลนส์ในท้องตลาดมักมีค่าระยะโฟกัสมาตรฐานตายตัว (เช่น 16mm, 25mm, 35mm) โค้ดนี้จะจำลองการปัดเศษจากค่าที่คำนวณได้ไปยังเลนส์มาตรฐานที่ใกล้เคียงที่สุด

**คำสั่งที่สำคัญ:**
- `min(stds, key=lambda s: abs(s - f_computed))`: เป็นท่ามาตรฐานในการเขียนโปรแกรม Python สำหรับการค้นหาค่าในลิสต์ (`stds`) ที่มีผลต่างเชิงสัมบูรณ์ (`abs`) จากตัวเลขเป้าหมายที่เราคำนวณไว้ (`f_computed`) น้อยที่สุด (อธิบายง่ายๆ คือการจับคู่หาเลนส์ที่เบอร์ใกล้เคียงที่สุดนั่นเอง)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `f_computed` | `20.0` | ได้เลนส์ 16 mm (ปัดลง) |
| | `56.3` | ได้เลนส์ 50 mm (ค่าในโค้ด — ปัดลงจาก 56.3) |
| `stds` | เพิ่ม/ลดค่าตามแคตตาล็อกผู้ผลิตจริง | ผลลัพธ์เปลี่ยนตามรุ่นเลนส์ที่หาซื้อได้จริงในโรงงาน |

**ผลของการปัดค่า — ต้องชดเชยที่หน้างาน**

| กรณี | ผลกระทบ | วิธีชดเชย |
|---|---|---|
| ปัด **ขึ้น** (เลนส์ยาวกว่าที่คำนวณ) | FOV **แคบกว่า** ที่ต้องการ อาจเห็นชิ้นงานไม่ครบ | ถอยกล้องออก (เพิ่ม WD) |
| ปัด **ลง** (เลนส์สั้นกว่าที่คำนวณ) | FOV **กว้างกว่า** ที่ต้องการ ความละเอียดต่อพิกเซลหยาบลง | ขยับกล้องเข้าใกล้ (ลด WD) |

> **ข้อควรระวัง:** เลนส์แต่ละรุ่นมีระยะโฟกัสใกล้สุด (MOD) ถ้าลด WD ต่ำกว่านั้นจะโฟกัสไม่เข้า ต้องใช้แหวนต่อ (extension ring) ช่วย


In [ ]:
f_computed = 56.3
stds = [4, 6, 8, 12, 16, 25, 35, 50, 75, 100]
pick = min(stds, key=lambda s: abs(s - f_computed))
print(f"Nearest Standard Lens: {pick} mm")



---

## 3. Image Processing (การประมวลผลภาพ)

### Slide 55: ฮิสโทแกรม
*คำอธิบาย:* ฮิสโทแกรม (Histogram) คือกราฟแสดงการกระจายตัวของความสว่างในภาพ (มืดไปสว่าง) หากกราฟมีภูเขาสองลูกแยกกันชัดเจน (Bimodal) แสดงว่าภาพนั้นเหมาะที่จะใช้วิธี Thresholding ในการแยกวัตถุออกจากพื้นหลัง

**ฟังก์ชันที่สำคัญ:**
- `cv2.calcHist()`: ใช้ประมวลผลคำนวณหาจำนวนพิกเซลในภาพที่มีระดับความสว่างต่างๆ (ตั้งแต่ 0 ถึง 255) เก็บไว้เป็นอาร์เรย์
- `plt.plot()`: ใช้ไลบรารี Matplotlib ของ Python เพื่อพล็อตกราฟเส้นแสดงผลฮิสโทแกรมออกมาให้เรามองเห็นได้ชัดเจน

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| จำนวน bin `[256]` | `[64]` | กราฟหยาบลง เห็นแนวโน้มภาพรวมง่ายขึ้น แต่ไม่เห็นรายละเอียดระดับสีเดี่ยว |
| | `[16]` | เหลือ 16 แท่ง เหมาะกับการดูคร่าวๆ ว่าภาพมืดหรือสว่างเกินไป |
| ช่วง `[0, 256]` | `[100, 200]` | ซูมดูเฉพาะช่วงความสว่างกลางภาพ (บริเวณผิวแผ่นงาน) |
| ช่องสีที่วัด `[gray], [0]` | `[img], [2]` | ทำฮิสโทแกรมของช่องสีแดงแทน (0=B, 1=G, 2=R) |

**วิธีอ่านกราฟเพื่อตัดสินใจขั้นต่อไป**

| รูปกราฟ | แปลว่า | ทำอะไรต่อ |
|---|---|---|
| ภูเขา 2 ลูกแยกชัด (Bimodal) | วัตถุกับพื้นหลังต่างกันชัด | ใช้ Otsu Threshold ได้เลย (สไลด์ 56) |
| ภูเขาลูกเดียวกว้าง | contrast ต่ำ | **แก้ที่ไฟและเลนส์ก่อน** ไม่ใช่ที่โค้ด |
| ค่ากองชิดขอบ 0 หรือ 255 | ภาพมืด/สว่างเกิน (ข้อมูลสูญไปแล้ว) | ปรับ exposure และ gain ของกล้อง |


In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hist = cv2.calcHist([gray], [0], None, [256], [0,256])
    plt.plot(hist)
    plt.title("Image Histogram")
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Frequency")
    plt.show()



### Slide 56: Thresholding
*คำอธิบาย:* การทำ Thresholding คือการแปลงภาพสีเทาให้เป็นภาพขาวดำ (Binary Image) วิธีของ Otsu จะช่วยคำนวณหาจุดตัด (Threshold value) ที่ดีที่สุดให้โดยอัตโนมัติจากค่าความแปรปรวนในฮิสโทแกรม ทำให้ไม่ต้องมานั่งเดาตัวเลข 127 เองเสมอไป

**ฟังก์ชันที่สำคัญ:**
- `cv2.threshold()`: ทำการเปลี่ยนภาพสีเทาเป็นภาพ Binary โดยเราสามารถระบุรูปแบบผ่าน Flag ได้ เช่น แบบค่าคงที่ปกติ (`THRESH_BINARY`) หรือสั่งให้คำนวณค่าอัตโนมัติแบบ Otsu (`THRESH_OTSU`)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| ค่า threshold `127` | `100` | รับพิกเซลเข้ามาเป็นสีขาวมากขึ้น วัตถุอ้วนขึ้น อาจติดเงา |
| | `180` | เข้มงวดขึ้น วัตถุผอมลง อาจขาดส่วนที่โดนเงา |
| Flag | `cv2.THRESH_BINARY` | สว่างกว่าค่า threshold → ขาว |
| | `cv2.THRESH_BINARY_INV` | **กลับด้าน** — เข้มกว่าค่า threshold → ขาว ใช้เมื่อวัตถุเข้มกว่าพื้นหลัง |
| | `+ cv2.THRESH_OTSU` | ให้คำนวณค่าที่ดีที่สุดเอง (ค่าที่ใส่ในช่องแรกจะถูกละทิ้ง จึงนิยมใส่ 0) |

**เมื่อ threshold ค่าเดียวเอาไม่อยู่ — ใช้ Adaptive แทน**

```python
adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, blockSize=51, C=5)
```

| พารามิเตอร์ | ผลที่ได้ |
|---|---|
| `blockSize` (ต้องเป็นเลขคี่) | ขนาดพื้นที่ที่ใช้คำนวณค่าเฉพาะจุด — เล็กเกินไปจะไวต่อ noise ใหญ่เกินไปจะกลายเป็น threshold ธรรมดา |
| `C` | ค่าที่หักออกจากค่าเฉลี่ย — เพิ่มขึ้นแล้วผลลัพธ์สะอาดขึ้นแต่วัตถุผอมลง |

> **หลักในงานจริง:** ถ้าต้องขยับค่า threshold บ่อยๆ เมื่อเปลี่ยนกะหรือเปลี่ยนล็อต แสดงว่าปัญหาอยู่ที่ **ไฟส่องสว่างไม่นิ่ง** ควรแก้ที่ไฟ (สไลด์ 43–51) ไม่ใช่ไล่แก้ตัวเลขในโค้ด


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, bin_fixed = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    _, bin_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    print("Fixed Threshold:")
    cv2_imshow(bin_fixed)
    print("Otsu Threshold:")
    cv2_imshow(bin_otsu)



### Slide 57: Filtering (Blur)
*คำอธิบาย:* ภาพจากกล้องมักมีสัญญาณรบกวน (Noise) การทำ Gaussian Blur จะช่วยเกลี่ยพิกเซลให้เนียนขึ้นแต่ขอบวัตถุอาจจะเบลอ ในขณะที่ Bilateral Filter จะช่วยลด Noise ได้โดยที่ยังรักษาเส้นขอบ (Edges) ของวัตถุไว้ให้คมชัด

**ฟังก์ชันที่สำคัญ:**
- `cv2.GaussianBlur()`: ทำการเบลอภาพด้วยสมการคณิตศาสตร์การแจกแจงแบบระฆังคว่ำ เหมาะสำหรับการเกลี่ยภาพให้เนียน
- `cv2.medianBlur()`: ทำการเบลอโดยหาค่ามัธยฐานในพื้นที่รอบๆ เหมาะกับการลบ Noise แบบเม็ดจุด (Salt and pepper)
- `cv2.bilateralFilter()`: เบลอโดยพิจารณาทั้งระยะห่างและค่าสี ทำให้เบลอเฉพาะพื้นที่เรียบ แต่ยังคงเส้นขอบคมไว้

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| ฟังก์ชัน | พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|---|
| `GaussianBlur` | ksize `(5,5)` | `(3,3)` | เบลอน้อย เก็บรายละเอียดไว้มาก ลด noise ได้น้อย |
| | | `(15,15)` | เบลอมาก noise หายเกลี้ยง แต่ขอบวัตถุเบลอตามจนวัดขนาดเพี้ยน |
| | sigma `0` | `0` | ให้ OpenCV คำนวณจาก ksize ให้เอง (แนะนำ) |
| `medianBlur` | ksize `5` | `3` / `9` | ยิ่งใหญ่ยิ่งลบจุด **salt & pepper** ได้ดี แต่รายละเอียดเล็กหายไปด้วย |
| `bilateralFilter` | `d = 9` | `15` | พื้นที่พิจารณากว้างขึ้น เบลอแรงขึ้น และ **ช้าลงมาก** |
| | `sigmaColor = 75` | `150` | ยอมเฉลี่ยสีที่ต่างกันมากขึ้น → เริ่มเบลอข้ามขอบ (เสียจุดเด่นของฟิลเตอร์นี้) |
| | `sigmaSpace = 75` | `150` | ดึงพิกเซลไกลๆ มาเฉลี่ยด้วย เบลอกว้างขึ้น |

**ข้อบังคับ:** `ksize` ต้องเป็น **เลขคี่** เสมอ (3, 5, 7, 9, …) ใส่เลขคู่จะ error

**เลือกฟิลเตอร์ตามชนิด noise**

| อาการที่เห็น | ใช้ฟิลเตอร์ |
|---|---|
| ภาพซ่าเป็นเม็ดทั่วภาพ | `GaussianBlur` |
| จุดขาว/ดำประปราย (เม็ดฝุ่น พิกเซลเสีย) | `medianBlur` |
| ต้องลด noise แต่ **ห้ามให้ขอบเบลอ** เพราะจะนำไปวัดขนาด | `bilateralFilter` |

> **ข้อควรระวัง:** เบลอแรงเกินไปทำให้ขอบขยับ ส่งผลให้ขนาดที่วัดได้ในสไลด์ 69 คลาดเคลื่อน — ใช้แรงเท่าที่จำเป็นเท่านั้น


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    blur_gauss = cv2.GaussianBlur(gray, (5, 5), 0)
    blur_median = cv2.medianBlur(gray, 5)
    blur_bilateral = cv2.bilateralFilter(gray, 9, 75, 75)

    print("Gaussian Blur:")
    cv2_imshow(blur_gauss)
    print("Bilateral Blur (Keeps edges):")
    cv2_imshow(blur_bilateral)



### Slide 58: Morphology

*คำอธิบาย:* Morphology เป็นเทคนิคปรับปรุงรูปทรงของวัตถุในภาพขาวดำ (Binary) โดยใช้ "บล็อกจำลอง" (Kernel) กวาดไปทั่วภาพ

ในหัวข้อนี้เราจะทำ Binary ที่ดึงเอา **ส่วนที่เข้มของภาพ** ออกมาเป็นสีขาว ได้แก่ ขอบแผ่นงาน · วงแหวนรูยึด 4 มุม · ขอบปุ่มกด · **ตัวอักษร `LOT A17`** · รอยขีดข่วน ส่วนจุดขาวกลางรูยึดจะกลายเป็น **รูสีดำ** อยู่ในวัตถุสีขาว ทำให้เห็นผลของแต่ละปฏิบัติการได้ชัดเจน

| บล็อก | ปฏิบัติการ | Kernel | ผลที่ควรเห็น |
|---|---|---|---|
| 58-0 | เตรียมภาพ Binary | — | ต้องรันก่อนเสมอ |
| 58-1 | **Dilation (ขยาย)** | 9×9 | เส้นและตัวอักษรอ้วนขึ้น รูดำเล็กลง |
| 58-2 | **Erosion (หด)** | 9×9 | เส้นบางลง ตัวอักษรและรอยขีดข่วนหายไป แต่ปุ่มก็หดเล็กลงด้วย |
| 58-3 | **Opening** = หด → ขยาย | 9×9 | **ตัวอักษร `LOT A17` และรอยขีดข่วนหายไป** แต่ขอบแผ่นและปุ่มยังคงขนาดเดิม |
| 58-4 | **Closing** = ขยาย → หด | 19×19 | **รูสีดำกลางรูยึดทั้ง 4 มุมถูกอุดจนตัน** โดยขนาดวัตถุยังคงเดิม |
| 58-5 | **Gradient** = ขยาย − หด | 5×5 | เหลือเฉพาะเส้นขอบของทุกวัตถุ |

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

พารามิเตอร์เดียวที่ต้องจูนในหัวข้อนี้คือ **ขนาดของ Kernel** และหลักคือ *Kernel ต้องใหญ่กว่าสิ่งที่ต้องการจัดการ*

| ปฏิบัติการ | Kernel | ผลที่ได้ (วัดจาก `part.jpg` จริง) |
|---|---|---|
| Opening | 5×5 | ตัวอักษรยัง **ไม่หาย** (เหลือ 1,164 px) เพราะเล็กกว่าความหนาตัวอักษร |
| Opening | **9×9** | ตัวอักษรหายหมด (เหลือ 0 px) ขอบแผ่นและปุ่มยังครบ ← ค่าที่ใช้ในบล็อก 58-3 |
| Opening | 15×15 | ตัวอักษรหาย แต่ขอบแผ่นและวงแหวนรูยึดเริ่มถูกลบไปด้วย |
| Closing | 15×15 | รูยึด **ยังไม่ถูกอุด** เพราะเล็กกว่าเส้นผ่านศูนย์กลางรู (19 px) |
| Closing | **19×19** | รูยึดถูกอุดตันพอดี วัตถุอื่นยังแยกจากกัน ← ค่าที่ใช้ในบล็อก 58-4 |
| Closing | 25×25 | อุดจุดกลางปุ่ม (23 px) ได้ด้วย แต่รูยึดเริ่ม **เชื่อมติดกับขอบแผ่น** |

| พารามิเตอร์อื่น | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| รูปทรง Kernel | `cv2.MORPH_ELLIPSE` | วงรี — รักษาความโค้งของชิ้นงาน (ค่าที่ใช้ในบล็อกเหล่านี้) |
| | `cv2.MORPH_RECT` | สี่เหลี่ยม — มุมของวัตถุจะเป็นเหลี่ยมตามไปด้วย เหมาะกับชิ้นงานขอบตรง |
| | `cv2.MORPH_CROSS` | กากบาท — กระทบภาพน้อยที่สุด ใช้เมื่อต้องการแตะภาพเบาๆ |
| `iterations` ใน `dilate`/`erode` | `2` | ทำซ้ำ 2 รอบ ให้ผลใกล้เคียงการใช้ Kernel ที่ใหญ่ขึ้นราวเท่าตัว แต่คุมทิศทางได้ละเอียดกว่า |

**สรุปหลักการเลือก**

| จะทำอะไร | ใช้ | ขนาด Kernel |
|---|---|---|
| ลบจุด/ตัวอักษร/รอยขีดข่วนออก โดยของใหญ่ต้องคงขนาดเดิม | Opening | **ใหญ่กว่าความหนา** ของสิ่งที่จะลบ |
| อุดรู/เชื่อมเส้นขาด โดยของใหญ่ต้องคงขนาดเดิม | Closing | **ใหญ่กว่าเส้นผ่านศูนย์กลางรู** |
| อยากให้วัตถุอ้วนขึ้น/ผอมลงจริงๆ | Dilation / Erosion | ตามระยะที่ต้องการขยายหรือหด (รัศมี ≈ ครึ่งหนึ่งของ Kernel) |

> **ข้อควรระวัง:** Kernel ใหญ่เกินไปทำให้วัตถุคนละชิ้น **เกาะติดกัน** ซึ่งจะทำให้ขั้นตอนนับชิ้นงานและวัดขนาดต่อจากนี้ผิดทั้งหมด — จูนแล้วต้องดูภาพผลลัพธ์ทุกครั้ง ไม่ใช่ดูแค่ตัวเลข

**ฟังก์ชันที่สำคัญ:**
- `cv2.getStructuringElement(shape, ksize)`: สร้าง Kernel ตามรูปทรงและขนาดที่ต้องการ
- `cv2.dilate()` / `cv2.erode()`: ปฏิบัติการขยายและหดพื้นฐาน
- `cv2.morphologyEx()`: เรียกปฏิบัติการผสมผ่านพารามิเตอร์ `MORPH_OPEN`, `MORPH_CLOSE`, `MORPH_GRADIENT`


In [ ]:
# Slide 58-0 : เตรียมภาพ Binary  (ต้องรันบล็อกนี้ก่อนบล็อก 58-1 ถึง 58-5)
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is None:
    print("Error: ไม่พบไฟล์ภาพ (กรุณาอัปโหลด part.jpg เข้า Colab ก่อน)")
else:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ใช้ค่า threshold คงที่ 120 กับ THRESH_BINARY_INV เพื่อให้ "ส่วนที่เข้ม" เป็นสีขาว
    # ได้แก่ ขอบแผ่นงาน วงแหวนรูยึด ขอบปุ่ม ตัวอักษร LOT A17 และรอยขีดข่วน
    # (ต่างจากสไลด์ 56 ที่ใช้ Otsu ซึ่งจะแยกแค่ "แผ่นงาน vs พื้นหลัง")
    _, bw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)

    # พิกัดสำหรับซูมดูผลลัพธ์ในบล็อกถัดๆ ไป
    TEXT_ROI = (280, 340, 340, 530)   # y1, y2, x1, x2 : บริเวณตัวอักษร LOT A17
    HOLE_ROI = (100, 172, 122, 194)   # y1, y2, x1, x2 : รูยึดมุมบนซ้าย

    def zoom(image, roi, scale=3):
        y1, y2, x1, x2 = roi
        crop = image[y1:y2, x1:x2]
        return cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST)

    print("Binary image : สีขาว = ส่วนที่เข้ม / จุดดำกลางรูยึด = 'รู' ที่จะใช้ทดสอบ Closing")
    cv2_imshow(bw)
    print("ซูมบริเวณตัวอักษร LOT A17 (ใช้ทดสอบ Opening):")
    cv2_imshow(zoom(bw, TEXT_ROI))
    print("ซูมบริเวณรูยึดมุมบนซ้าย (ใช้ทดสอบ Closing):")
    cv2_imshow(zoom(bw, HOLE_ROI))


In [ ]:
# Slide 58-1 : Dilation (ขยาย)
import cv2
from google.colab.patches import cv2_imshow

if 'bw' not in globals():
    print("กรุณารันบล็อก 58-0 (เตรียมภาพ Binary) ก่อน")
else:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    dilated = cv2.dilate(bw, kernel, iterations=1)

    print("Dilation - เส้นและตัวอักษรอ้วนขึ้น รูดำเล็กลง:")
    cv2_imshow(dilated)
    print("ซูมตัวอักษร (ตัวหนาขึ้น):")
    cv2_imshow(zoom(dilated, TEXT_ROI))


In [ ]:
# Slide 58-2 : Erosion (หด)
import cv2
from google.colab.patches import cv2_imshow

if 'bw' not in globals():
    print("กรุณารันบล็อก 58-0 (เตรียมภาพ Binary) ก่อน")
else:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    eroded = cv2.erode(bw, kernel, iterations=1)

    print("Erosion - ตัวอักษรและรอยขีดข่วนหายไป แต่สังเกตว่าปุ่มและขอบแผ่นก็หดเล็กลงด้วย:")
    cv2_imshow(eroded)
    print("ซูมตัวอักษร (หายไปแล้ว):")
    cv2_imshow(zoom(eroded, TEXT_ROI))


In [ ]:
# Slide 58-3 : Opening = Erosion -> Dilation  (ลบสิ่งเล็กๆ ออก)
import cv2
from google.colab.patches import cv2_imshow

if 'bw' not in globals():
    print("กรุณารันบล็อก 58-0 (เตรียมภาพ Binary) ก่อน")
else:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    opening = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)

    # ยืนยันเชิงตัวเลขว่าตัวอักษรหายไปจริง
    y1, y2, x1, x2 = TEXT_ROI
    before = int((bw[y1:y2, x1:x2] > 0).sum())
    after  = int((opening[y1:y2, x1:x2] > 0).sum())
    print(f"พิกเซลตัวอักษร : ก่อน Opening = {before} px -> หลัง Opening = {after} px")

    print("Opening - ตัวอักษร LOT A17 และรอยขีดข่วนหายไป แต่ขอบแผ่นและปุ่มยังคงขนาดเดิม")
    print("(ต่างจาก Erosion ที่ลบตัวอักษรได้เหมือนกัน แต่ทำให้ทุกอย่างหดเล็กลงไปด้วย)")
    cv2_imshow(opening)
    print("ซูมบริเวณตัวอักษร (ควรเหลือแต่พื้นดำ):")
    cv2_imshow(zoom(opening, TEXT_ROI))


In [ ]:
# Slide 58-4 : Closing = Dilation -> Erosion  (อุดรูดำ)
import cv2
from google.colab.patches import cv2_imshow

if 'bw' not in globals():
    print("กรุณารันบล็อก 58-0 (เตรียมภาพ Binary) ก่อน")
else:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19))
    closing = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)

    # ยืนยันเชิงตัวเลขว่ารูถูกอุดจริง (ตรวจค่าพิกเซลที่จุดกึ่งกลางรูยึดมุมบนซ้าย)
    cy, cx = 135, 150
    print(f"พิกเซลกลางรูยึด : ก่อน Closing = {bw[cy, cx]} (ดำ) -> หลัง Closing = {closing[cy, cx]} (ขาว = ถูกอุดแล้ว)")

    print("Closing - รูดำกลางรูยึดทั้ง 4 มุมถูกอุดจนตัน โดยขนาดวัตถุยังคงเดิม")
    print("(ต่างจาก Dilation ที่อุดรูได้เหมือนกัน แต่ทำให้ทุกอย่างอ้วนขึ้นไปด้วย)")
    cv2_imshow(closing)
    print("ซูมบริเวณรูยึดมุมบนซ้าย (ควรกลายเป็นวงกลมทึบ):")
    cv2_imshow(zoom(closing, HOLE_ROI))

    # หมายเหตุ : จุดดำกลางปุ่มกดกว้างราว 23 px ซึ่งใหญ่กว่า Kernel 19x19 จึงยังไม่ถูกอุด
    # ลองเปลี่ยนเป็น (25, 25) จะเห็นจุดกลางปุ่มถูกอุดตามไปด้วย แต่รูยึดจะเริ่ม
    # เชื่อมติดกับขอบแผ่น -- Kernel ใหญ่เกินไปทำให้วัตถุคนละชิ้นเกาะติดกัน


In [ ]:
# Slide 58-5 : Gradient = Dilation - Erosion  (หาเส้นขอบ)
import cv2
from google.colab.patches import cv2_imshow

if 'bw' not in globals():
    print("กรุณารันบล็อก 58-0 (เตรียมภาพ Binary) ก่อน")
else:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    grad = cv2.morphologyEx(bw, cv2.MORPH_GRADIENT, kernel)

    print("Morphological Gradient - เหลือเฉพาะเส้นขอบของทุกวัตถุ:")
    cv2_imshow(grad)




### Slide 60: Canny Edge Detection
*คำอธิบาย:* อัลกอริทึมของ Canny เป็นเครื่องมือมาตรฐานในการหาเส้นขอบ (Edges) โดยจะตรวจจับบริเวณที่มีการเปลี่ยนแปลงความสว่างอย่างฉับพลัน ซึ่งเป็นจุดเด่นสำคัญของโครงสร้างวัตถุ

**ฟังก์ชันที่สำคัญ:**
- `cv2.Canny()`: ฟังก์ชันหาเส้นขอบแบบสำเร็จรูป โดยจะรับพารามิเตอร์ Lower Threshold (100) และ Upper Threshold (200) เพื่อช่วยในการตัดสินใจว่าขอบเขตความสว่างที่ต่างกันระดับไหนถึงจะถูกนับเป็น "เส้นขอบ"

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

`cv2.Canny(image, threshold1, threshold2)` — ค่าต่ำใช้ตัดสินขอบอ่อน ค่าสูงใช้ตัดสินขอบชัด

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `(50, 150)` | ค่าต่ำ | ขอบเยอะขึ้น รวมขอบจางและ noise เข้ามาด้วย |
| `(100, 200)` | ค่าที่ใช้ในโค้ด | สมดุล — เป็นจุดตั้งต้นที่แนะนำ |
| `(150, 300)` | ค่าสูง | เหลือเฉพาะขอบที่ชัดจริง ขอบจางหายไป เส้นอาจขาดตอน |
| ภาพที่ป้อนเข้า | `gray` (ไม่ผ่าน blur) | ขอบรกขึ้นมากเพราะ noise ทุกเม็ดถูกนับเป็นขอบ — **ควร blur ก่อนเสมอ** |
| ksize ของ blur ก่อนหน้า | `(3,3)` → `(9,9)` | เบลอแรงขึ้น ขอบเหลือน้อยลงและเลื่อนตำแหน่งเล็กน้อย |

**กติกาที่ควรจำ:** ตั้งอัตราส่วน `threshold2 : threshold1` ไว้ที่ประมาณ **2:1 ถึง 3:1** แล้วค่อยเลื่อนทั้งคู่ขึ้น-ลงพร้อมกัน จะจูนง่ายกว่าการขยับทีละตัว

> **สังเกตจากภาพตัวอย่างนี้:** `part.jpg` เป็นภาพที่ขอบคมมาก ผลลัพธ์ที่ (50,150) กับ (100,200) จึงแทบไม่ต่างกัน (13,508 พิกเซลขอบเท่ากัน) แต่กับภาพถ่ายจากกล้องจริงที่มี noise จะเห็นความต่างชัดเจน


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(blur, 100, 200)   # อัตราส่วน 1:2 ตามที่สไลด์แนะนำ
    cv2_imshow(edges)



### Slide 62: Contour Detection
*คำอธิบาย:* คอนทัวร์ (Contour) คือการลากเส้นเชื่อมต่อพิกเซลขอบที่อยู่ติดกันให้กลายเป็นรูปทรงปิด ทำให้เราสามารถคำนวณพื้นที่ (Area), หาจุดศูนย์กลาง, และวาดกล่องครอบ (Bounding Box) วัตถุแต่ละชิ้นแยกออกจากกันได้

**ฟังก์ชันที่สำคัญ:**
- `cv2.findContours()`: ทำหน้าที่สกัดดึงข้อมูลพิกัดโครงร่างของวัตถุสีขาวในภาพ Binary ออกมาเก็บเป็น List
- `cv2.drawContours()`: สั่งวาดเส้นโครงร่างลงบนภาพสีเพื่อนำไปแสดงผลดูด้วยตา
- `cv2.contourArea()`: คำนวณพื้นที่ (หาจำนวนพิกเซล) ที่อยู่ภายในโครงร่างนั้นๆ
- `cv2.boundingRect()`: ตีกรอบกล่องสี่เหลี่ยมแนวตั้ง-แนวนอน ล้อมรอบพิกัดของโครงร่างวัตถุที่เจอ

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| เกณฑ์ `area > 100` | `> 1000` | กรอง noise ได้มากขึ้น เหลือเฉพาะชิ้นงานใหญ่ |
| | `> 10` | ติดจุดรบกวนเล็กๆ เข้ามาเป็นชิ้นงานปลอมจำนวนมาก |
| โหมด `cv2.RETR_EXTERNAL` | `cv2.RETR_LIST` | ได้ **คอนทัวร์ด้านในด้วย** เช่น ขอบรูยึดและขอบปุ่ม (จำนวนคอนทัวร์เพิ่มขึ้นมาก) |
| | `cv2.RETR_TREE` | ได้ทั้งหมดพร้อมโครงสร้างลำดับชั้น (รู้ว่ารูไหนอยู่ในชิ้นงานไหน) ใช้ตรวจ "รูหาย" ได้ |
| วิธีเก็บจุด `CHAIN_APPROX_SIMPLE` | `CHAIN_APPROX_NONE` | เก็บทุกพิกเซลบนเส้นขอบ ใช้หน่วยความจำมากกว่าหลายเท่า แต่ได้เส้นละเอียดครบ |
| ฟังก์ชันวัดกรอบ | `cv2.boundingRect(c)` | กรอบล็อกแนวตั้ง-นอน (ค่าในโค้ด) — ง่ายแต่ถ้าชิ้นงานเอียงจะได้ขนาดเกินจริง |
| | `cv2.minAreaRect(c)` | กรอบหมุนตามชิ้นงานได้ — แม่นกว่าเมื่อวางเอียง (ใช้ในสไลด์ 69) |

**ฟังก์ชันวัดอื่นที่ใช้ต่อยอดได้**

| คำสั่ง | ได้อะไร | ใช้ทำอะไร |
|---|---|---|
| `cv2.arcLength(c, True)` | ความยาวเส้นรอบรูป | คู่กับพื้นที่เพื่อคำนวณความกลม |
| `4*np.pi*area/(peri**2)` | ค่าความกลม (1.0 = วงกลมสมบูรณ์) | **คัดแยกชิ้นดี/ชิ้นผิดรูป** |
| `cv2.convexHull(c)` | รูปทรงหุ้มด้านนอก | ตรวจรอยแหว่ง/รอยบิ่นที่ขอบ |

> **หลักในงานจริง:** ตั้งเกณฑ์ `area` จากขนาดชิ้นงานจริงในหน่วยพิกเซล — คำนวณจาก `พื้นที่จริง (mm²) ÷ (mm_per_px)²` แล้วเผื่อขอบเขตบน-ล่างสัก ±20%


In [ ]:
import cv2
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    img_contours = img.copy()
    cv2.drawContours(img_contours, cnts, -1, (0, 255, 0), 2)

    for i, c in enumerate(cnts):
        area = cv2.contourArea(c)
        if area > 100:
            x, y, w, h = cv2.boundingRect(c)
            print(f"Contour {i}: Area={area}")
            cv2.rectangle(img_contours, (x, y), (x+w, y+h), (0, 0, 255), 2)

    cv2_imshow(img_contours)



### Slide 64: Hough Circles
*คำอธิบาย:* Hough Transform เป็นเทคนิคทางคณิตศาสตร์ที่ใช้ "โหวต" หาพิกเซลที่ประกอบกันเป็นรูปทรงเรขาคณิต (เช่น วงกลม หรือเส้นตรง) ทำให้สามารถตรวจจับวงกลมได้แม่นยำแม้ว่าวงกลมนั้นจะแหว่งไปบางส่วนก็ตาม

**ฟังก์ชันที่สำคัญ:**
- `cv2.HoughCircles()`: อัลกอริทึมค้นหารูปทรงวงกลมในภาพ โดยเราต้องกำหนดพารามิเตอร์เพื่อจำกัดขอบเขตการค้นหา เช่น รัศมีขั้นต่ำ (`minRadius`) และสูงสุด (`maxRadius`) รวมถึงความเข้มงวดในการตรวจจับ (`param1`, `param2`)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | หน้าที่ | ปรับแล้วได้อะไร |
|---|---|---|
| `minDist` | ระยะห่างขั้นต่ำระหว่างจุดศูนย์กลางวงกลม | **ตัวสำคัญที่สุด** — เล็กเกินไปจะได้วงกลมซ้อนทับกันหลายวงบนวัตถุเดียว ตั้งประมาณรัศมีวัตถุที่เล็กที่สุด |
| `param2` | เกณฑ์การโหวต | ต่ำ = เจอเยอะแต่มีวงปลอม / สูง = เจอน้อยแต่มั่นใจ **ค่านี้จูนบ่อยที่สุด** |
| `param1` | ค่า threshold บนของ Canny ที่ใช้ภายใน | ต่ำ = รับขอบจางเข้ามาด้วย / สูง = ใช้เฉพาะขอบชัด |
| `minRadius`, `maxRadius` | จำกัดขนาดวงกลมที่ค้นหา | **ตั้งให้แคบเข้าไว้** ช่วยทั้งความเร็วและความแม่นยำ ตัดวงกลมปลอมได้มาก |
| `dp` | อัตราส่วนความละเอียดของ accumulator | `1` = ละเอียดเท่าภาพ (แนะนำ), `2` = หยาบลงครึ่งหนึ่ง เร็วขึ้นแต่ตำแหน่งคลาดเคลื่อน |

**ทดลองจริงกับ `part.jpg`** (ชิ้นงานนี้มีปุ่ม 6 ปุ่ม + รูยึด 4 รู = 10 วง)

| ค่าที่ตั้ง | ผลลัพธ์ |
|---|---|
| `minDist=20, param2=30, r=(5,100)` | **22 วง** — เกินจริงมาก เพราะ `minDist` แคบ ทำให้เกิดวงซ้อนบนปุ่มเดียวกัน (ค่าเดิมในโค้ด) |
| `minDist=40, param2=30, r=(5,100)` | **10 วง** — ตรงกับของจริงพอดี |
| `minDist=40, param2=40, r=(10,80)` | **6 วง** — เข้มงวดเกินไป รูยึดขนาดเล็กหลุดหายไป |

> **ลำดับการจูนที่แนะนำ:** ล็อก `minRadius`/`maxRadius` จากขนาดจริงก่อน → ตั้ง `minDist` ให้ใกล้เคียงรัศมีวัตถุ → ค่อยไล่ `param2` ขึ้นทีละ 5 จนจำนวนวงกลมนิ่งและตรงกับความจริง


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    blur_for_circles = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(blur_for_circles, cv2.HOUGH_GRADIENT, dp=1, minDist=20,
                               param1=100, param2=30, minRadius=5, maxRadius=100)

    img_circles = img.copy()
    if circles is not None:
        circles = np.uint16(np.around(circles))
        print(f"Detected {len(circles[0])} circles")
        for i in circles[0, :]:
            cv2.circle(img_circles, (i[0], i[1]), i[2], (0, 255, 0), 2)
            cv2.circle(img_circles, (i[0], i[1]), 2, (0, 0, 255), 3)

    cv2_imshow(img_circles)



---

## 4. Measurement & Inspection (การตรวจสอบและวัดขนาด)

### Slide 67: Pixel Calibration
*คำอธิบาย:* ก่อนที่เราจะวัดขนาดชิ้นงานเป็นหน่วยมิลลิเมตร (mm) ได้ เราต้องเทียบสัดส่วน (Calibrate) ก่อนว่า 1 พิกเซลในภาพมีขนาดเท่ากับกี่มิลลิเมตรในโลกจริง โดยการใช้ชิ้นงานอ้างอิงที่มีขนาดแน่นอน

**ตัวแปรที่สำคัญ:**
- `mm_per_px`: เราหาค่าสัมประสิทธิ์คงที่ตัวนี้มาเก็บไว้ สำหรับใช้คูณแปลงหน่วยความยาวจาก "พิกเซล" กลับไปเป็น "มิลลิเมตร" ในตอนท้าย (ได้จากการนำความยาวจริงหารด้วยพิกเซลที่ใช้วัดในภาพอ้างอิง)

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `known_mm` | ความยาวจริงของชิ้นอ้างอิงที่ใช้ | ต้องเป็นค่าที่วัดมาแล้วจริง เช่น เกจบล็อกหรือแท่งสอบเทียบ |
| `measured_px` | จำนวนพิกเซลที่วัดได้จากภาพชิ้นอ้างอิงนั้น | ยิ่งวัดจากชิ้นอ้างอิงที่ **ยาว** ยิ่งแม่น เพราะความผิดพลาด ±1 px กระทบน้อยลง |

**ผลของการเปลี่ยนค่า**

| ถ้า… | `mm_per_px` | ผลต่อขนาดที่วัดได้ |
|---|---|---|
| `measured_px` เพิ่มเป็น 800 (ภาพละเอียดขึ้น 2 เท่า) | 0.0625 | วัดได้ละเอียดขึ้น 2 เท่า |
| `known_mm` ผิดไป 1 mm (50 → 51) | 0.1275 | ขนาดทุกชิ้นที่วัดต่อจากนี้ **เพี้ยนไป 2% ทั้งหมด** |

**ความแม่นยำที่ทำได้:** ความผิดพลาด ±1 พิกเซล = ±`mm_per_px` มิลลิเมตร → ที่ค่า 0.125 mm/px หมายความว่าวัดได้ละเอียดที่สุดราว ±0.125 mm ถ้าต้องการละเอียดกว่านี้ ต้องเพิ่มความละเอียดกล้องหรือลด FOV (กลับไปดูสไลด์ 40)

> **ข้อควรระวัง:** ต้อง calibrate **ใหม่ทุกครั้ง** ที่เปลี่ยนเลนส์ ขยับกล้อง หรือเปลี่ยนระยะทำงาน (WD) — และควรวางชิ้นอ้างอิงที่ระนาบความสูงเดียวกับชิ้นงานจริง มิฉะนั้นจะเกิด perspective error (เว้นแต่ใช้เลนส์ telecentric ตามสไลด์ 39)


In [ ]:
known_mm = 50.0
measured_px = 400.0

mm_per_px = known_mm / measured_px
print(f"Calibration Factor: {mm_per_px} mm/pixel")



### Slide 69: Measurement
*คำอธิบาย:* เมื่อรวมเทคนิคทั้งหมดเข้าด้วยกัน: เราหาคอนทัวร์ -> หากล่องที่ครอบวัตถุแบบแนบสนิทที่สุด (minAreaRect) -> หาความกว้างยาวระดับพิกเซล -> นำไปคูณกับค่า Calibration Factor ก็จะได้ขนาดชิ้นงานจริงในหน่วยมิลลิเมตร

**ฟังก์ชันที่สำคัญ:**
- `cv2.minAreaRect()`: สร้างกรอบสี่เหลี่ยมรอบคอนทัวร์ให้มีพื้นที่น้อยที่สุด (กล่องแบบนี้สามารถหมุนเอียงตามทรงของวัตถุได้อย่างอิสระ แตกต่างจาก `boundingRect` ที่ล็อกแนวตั้งนอน) ซึ่งจะให้ผลลัพธ์พิกัดการวัดที่ตรงกับชิ้นงานและแม่นยำกว่า

**คำอธิบายพารามิเตอร์ที่เกี่ยวข้อง**

| พารามิเตอร์ | ลองเปลี่ยนเป็น | ผลที่ได้ |
|---|---|---|
| `cv2.minAreaRect(c)` | `cv2.boundingRect(c)` | กรอบล็อกแนวตั้ง-นอน — ถ้าชิ้นงานวางเอียง 30° ขนาดที่ได้จะ **เกินจริง** อย่างมาก |
| `max(cnts, key=cv2.contourArea)` | `for c in cnts:` วนทุกคอนทัวร์ | วัดทุกชิ้นในภาพแทนที่จะวัดเฉพาะชิ้นใหญ่สุด |
| `mm_per_px = 50.0/400.0` | ค่าที่ได้จริงจากสไลด์ 67 | ค่านี้คือตัวคูณสุดท้าย — ผิดที่นี่ ผิดทั้งรายงาน |
| `rect[2]` (ที่ยังไม่ได้ใช้) | `print(rect[2])` | ได้ **มุมเอียง** ของชิ้นงาน ใช้ตรวจการวางผิดแนวได้ทันที |
| ค่า threshold ในบรรทัดก่อนหน้า | เลื่อนขึ้น/ลง 10 ระดับ | ขอบวัตถุขยับ 1–2 px → ขนาดที่วัดได้เปลี่ยนราว 0.25 mm **นี่คือแหล่งความคลาดเคลื่อนอันดับหนึ่ง** |

**เพิ่มการตัดสิน OK/NG ต่อยอดได้ทันที**

```python
SPEC_W, SPEC_H, TOL = 57.0, 93.0, 0.5   # mm
ok = abs(w_mm - SPEC_W) <= TOL and abs(h_mm - SPEC_H) <= TOL
print("ผลตรวจ:", "OK" if ok else "NG")
```

| พารามิเตอร์ | ปรับแล้วได้อะไร |
|---|---|
| `TOL` แคบลง (เช่น 0.1) | เข้มงวดขึ้น แต่ถ้าแคบกว่าความละเอียดของระบบ (±`mm_per_px`) จะเกิด **NG ปลอม** เต็มไปหมด |
| `TOL` กว้างขึ้น | ปล่อยผ่านง่ายขึ้น เสี่ยงปล่อยของเสียหลุดไปถึงลูกค้า |

> **หลักในงานจริง:** ค่า tolerance ที่ตั้งได้ ต้องกว้างกว่าความละเอียดของระบบอย่างน้อย 3–5 เท่า — ระบบที่ให้ 0.125 mm/px ไม่ควรรับงานที่ต้องการ tolerance ±0.1 mm


In [ ]:
import cv2
import numpy as np

img = cv2.imread('part.jpg')
if img is not None:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if cnts:
        c = max(cnts, key=cv2.contourArea)
        rect = cv2.minAreaRect(c)
        w_px, h_px = rect[1]

        mm_per_px = 50.0 / 400.0
        w_mm = w_px * mm_per_px
        h_mm = h_px * mm_per_px

        print(f"Measured Size: {w_mm:.2f} x {h_mm:.2f} mm")